# Sigiriya Local Neo4j → SinLlama (Colab inference only)

This notebook does **not** connect to Neo4j. It takes a JSON file exported from local graph retrieval, builds grounded evidence context and uses SinLlama for Sinhala answer generation. For a live integration, connect an authenticated retrieval API later.

## Step 1 — T4 runtime and dependencies
Select **Runtime → Change runtime type → T4 GPU**.

In [1]:
%pip -q install "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1" safetensors

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch, json, re
print("CUDA available:",torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Choose T4 GPU under Runtime > Change runtime type")
print("GPU:",torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## Step 2 — Upload local retrieval result JSON
The file must contain a JSON object with `question`, `status`, `retrieved_chunks`, optional `graph_paths`, `extractive_answer`, and `hybrid_top3`. It can be produced on your laptop using the original group retrieval pipeline.

In [3]:
from google.colab import files
uploaded=files.upload()
json_names=[name for name in uploaded if name.lower().endswith(".json")]
assert json_names,"Upload one retrieval JSON file"
with open(json_names[0],encoding="utf-8") as f:
    payload=json.load(f)
assert isinstance(payload,dict) and isinstance(payload.get("retrieved_chunks"),list),"JSON must contain retrieved_chunks list"
question=payload.get("question","").strip()
assert question,"JSON must include question"
print("Question:",question)
print("Status:",payload.get("status","not supplied"))
print("Chunks:",len(payload["retrieved_chunks"]))

Saving sigiriya_retrieval_result.json to sigiriya_retrieval_result.json
Question: සීගිරියේ ජල උද්‍යානවල ජල මාර්ග ගැන විස්තර කරන්න.
Status: SOURCE_GROUNDED_EXTRACTIVE
Chunks: 3


## Step 3 — Build source evidence context

In [4]:
def build_evidence_context(payload,max_chars_each=1150):
    chunks=payload.get("retrieved_chunks") or []
    lines=[]
    for ch in chunks[:3]:
        lines.append(f"[{ch.get('id','unknown')}] {ch.get('heading','')}\n{ch.get('text','')[:max_chars_each].strip()}")
    evidence_ids={ch.get("id") for ch in chunks}
    for path in (payload.get("graph_paths") or [])[:3]:
        for edge in path.get("edges",[]):
            if edge.get("chunk_id") in evidence_ids and edge.get("quote"):
                line=f"[Graph source {edge['chunk_id']}] {edge.get('relation','')}: {edge['quote']}"
                if line not in lines:lines.append(line)
    return "\n\n".join(lines)

context=build_evidence_context(payload)
print(context[:1800] or "NO RETRIEVED CONTEXT")

[sigiriya_006] 6. ජල උද්‍යාන
සීගිරි පුරාවිද්‍යා සංකීර්ණයට ඇතුළු වූ පසු දැකගත හැකි ප්‍රධාන අංගයක් වන්නේ ජල උද්‍යානයි.

මෙහි ජල තටාක, ජල මාර්ග, මණ්ඩප, අංගන සහ සැලසුම් කළ උද්‍යාන දැකගත හැකිය.

ජලය පාලනය සහ බෙදාහැරීම සඳහා පැරණි කාලයේ භූගත ජල මාර්ග සහ වෙනත් ජල පද්ධති භාවිත කර ඇත.

මෙම උද්‍යාන කාශ්‍යප රජුගේ යුගයේ පැවති උසස් භූ දර්ශන නිර්මාණ සහ ඉංජිනේරු දැනුම පෙන්වයි.

[sigiriya_007] 7. දිය උල්පත් උද්‍යානය
සීගිරියේ ප්‍රධාන මාර්ගය දෙපස සමමිතිකව ඉදිකර ඇති දිය උල්පත් දැකගත හැකිය.

මෙම දිය උල්පත් වෙත භූගත ජල මාර්ග ඔස්සේ ජලය ලබාදී ඇත.

ගුරුත්වාකර්ෂණය සහ ජල පීඩනය භාවිත කරමින් මෙම පද්ධතිය ක්‍රියාත්මක වූ බව සැලකේ.

ප්‍රමාණවත් ජලය පවතින අවස්ථාවලදී පැරණි ජල පද්ධතියේ ඇතැම් කොටස් අදත් ක්‍රියාත්මක විය හැකිය.

[sigiriya_015] 15. සීගිරිය ආසන්න වනජීවී සෆාරි
සීගිරිය නැරඹීම සමඟ වනජීවී සෆාරි සංචාරයක්ද එක් කළ හැකිය.

මින්නේරිය සහ කවුඩුල්ල ජාතික වනෝද්‍යාන සීගිරියට උතුරින් පිහිටා ඇත.

මෙම ප්‍රදේශ අතර වන අලින්ට ගමන් කළ හැකි වනජීවී මාර්ග පවතී. කාලගුණය සහ වාරය අනුව අලි රංචු සිටින ප්‍රදේශ වෙනස් විය හැකිය.

සීගිරිය අවට

## Step 4 — Load SinLlama on GPU
The weights are a large download; GPU memory and storage are required. Never expose Neo4j credentials in this notebook.

In [5]:
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
MODEL_ID="SAWithanage/SinLlama-Llama-3-8B-Merged"
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
tok=AutoTokenizer.from_pretrained(MODEL_ID)
llm=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=quant,device_map="auto",low_cpu_mem_usage=True)
llm.eval()
if tok.pad_token_id is None:tok.pad_token=tok.eos_token
print("Model loaded")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/331 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

... (8 outputs left out)

model-00009-of-00009.safetensors:   0%|          | 0.00/1.38G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

Model loaded


## Step 5 — Generate and inspect Sinhala answer
This is not proof of factual correctness. Check every generated claim against the displayed evidence.

In [6]:
def generate_answer(question,payload,max_new_tokens=120):
    if payload.get("status") in ("LIVE_DATA_UNSUPPORTED","INSUFFICIENT_DOCUMENT_EVIDENCE"):
        return payload.get("extractive_answer", "මූලාශ්‍රවල ප්‍රමාණවත් තොරතුරු නොමැත."),"DECLINED_BY_RETRIEVAL"
    context=build_evidence_context(payload)
    if not context.strip():
        return "මූලාශ්‍රවල ප්‍රමාණවත් තොරතුරු නොමැත.","NO_RETRIEVED_EVIDENCE"
    prompt=("ඔබ සිංහල ප්‍රශ්නවලට මූලාශ්‍ර මත පදනම්ව පිළිතුරු සපයන සහායකයෙකි. "
            "පහත මූලාශ්‍රයේ ඇති තොරතුරු පමණක් භාවිත කරන්න. "
            "මූලාශ්‍රයේ නැති කරුණු එකතු නොකරන්න. කෙටියෙන් සිංහලෙන් පිළිතුරු දෙන්න.\n\n"
            f"මූලාශ්‍ර:\n{context}\n\nප්‍රශ්නය: {question}\nපිළිතුර:")
    batch=tok(prompt,return_tensors="pt",truncation=True,max_length=3000)
    batch={k:v.to(next(llm.parameters()).device) for k,v in batch.items()}
    with torch.inference_mode():
        generated=llm.generate(**batch,max_new_tokens=max_new_tokens,do_sample=False,repetition_penalty=1.15,pad_token_id=tok.eos_token_id,eos_token_id=tok.eos_token_id)
    text=tok.decode(generated[0][batch["input_ids"].shape[1]:],skip_special_tokens=True).strip()
    text=re.split(r"\n\s*(?:Question|ප්‍රශ්නය|Source|මූලාශ්‍ර)\s*:",text,maxsplit=1)[0].strip()
    return text or "(No usable output)","GENERATED_UNVERIFIED"

answer,status=generate_answer(question,payload)
print("QUESTION:",question)
print("\nRETRIEVAL ANSWER:",payload.get("extractive_answer",""))
print("\nSINLLAMA STATUS:",status)
print("SINLLAMA ANSWER:",answer)
print("\nSOURCE IDS:",payload.get("hybrid_top3",[]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: සීගිරියේ ජල උද්‍යානවල ජල මාර්ග ගැන විස්තර කරන්න.

RETRIEVAL ANSWER: සීගිරි පුරාවිද්‍යා සංකීර්ණයට ඇතුළු වූ පසු දැකගත හැකි ප්‍රධාන අංගයක් වන්නේ ජල උද්‍යානයි. මෙහි ජල තටාක, ජල මාර්ග, මණ්ඩප, අංගන සහ සැලසුම් කළ උද්‍යාන දැකගත හැකිය.

SINLLAMA STATUS: GENERATED_UNVERIFIED
SINLLAMA ANSWER: පුරාණ ශ්‍රී ලංකාවේ වාරිමාර්ග තාක්ෂණය ඉතා දියුණු මට්ටමක පැවතුනි.පුරාණ වැව් තැනීමේ දී ඒවා පිරවීම පිණිස යොදාගත් ක්‍රමෝපායන් ද ඒ හා සමාන ලෙස වැදගත් වේ.වැවක් ඉදිකිරීමේදී එහි හැඩය තීරණය කිරීම කෙරෙහි වැඩි අවධානයක් යොමු කරන ලදී.ඒවා අතරින් වඩාත් ප්‍රචලිත වූයේ "වටා වැව" යන නමයි.මේඅතර වට දාර සහිත විශාල ජලාශවලට වඩා කුඩා ප්‍රමාණයේ රවුම් හැඩැති ජලාශයන් පිළිබඳවද සඳහන් වී තිබේ.එහි ප්‍රතිඵලයක් වශයෙන් බොහෝ විට දක්නට ලැබුණේ තරමක් දුරට පැතලි මතුපිටකින් යුත් ජලාශ ය.කෙසේ වෙතත්

SOURCE IDS: ['sigiriya_006', 'sigiriya_007', 'sigiriya_015']


## Next step
To make this interactive, host your **local retrieval service** behind an authenticated HTTPS tunnel. Colab sends a question to that service, and the service returns exactly this JSON payload. Keep the database Bolt port private.

In [7]:

def generate_answer_v2(question, payload, max_new_tokens=90):
    status = payload.get("status", "")

    if status in ("LIVE_DATA_UNSUPPORTED", "INSUFFICIENT_DOCUMENT_EVIDENCE"):
        return "මූලාශ්‍රවල මෙම ප්‍රශ්නයට පිළිතුරු සැපයීමට ප්‍රමාණවත් තොරතුරු නොමැත.", "DECLINED_BY_RETRIEVAL"

    chunks = payload.get("retrieved_chunks", [])
    if not chunks:
        return "මූලාශ්‍රවල ප්‍රමාණවත් තොරතුරු නොමැත.", "NO_EVIDENCE"

    # Begin with a smaller context to reduce topic drift.
    context = "\n\n".join(
        f"{ch.get('heading', '')}\n{ch.get('text', '')[:650]}"
        for ch in chunks[:2]
    )

    prompt = f"""පහත පාඨය කියවා ප්‍රශ්නයට පිළිතුරු දෙන්න.
පාඨයේ නොමැති තොරතුරු එකතු නොකරන්න.
ප්‍රශ්නයට අදාළ කරුණු පමණක් සිංහලෙන් වාක්‍ය 2-3කින් ලියන්න.

පාඨය:
{context}

ප්‍රශ්නය: {question}

පිළිතුර:"""

    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )
    inputs = {
        k: v.to(next(llm.parameters()).device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tok.eos_token_id,
            eos_token_id=tok.eos_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tok.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer, "GENERATED_UNVERIFIED"


answer_v2, status_v2 = generate_answer_v2(question, payload)

print("QUESTION:", question)
print("\nRETRIEVAL ANSWER:", payload.get("extractive_answer", ""))
print("\nSINLLAMA V1:", answer)
print("\nSINLLAMA V2:", answer_v2)
print("\nSTATUS:", status_v2)


QUESTION: සීගිරියේ ජල උද්‍යානවල ජල මාර්ග ගැන විස්තර කරන්න.

RETRIEVAL ANSWER: සීගිරි පුරාවිද්‍යා සංකීර්ණයට ඇතුළු වූ පසු දැකගත හැකි ප්‍රධාන අංගයක් වන්නේ ජල උද්‍යානයි. මෙහි ජල තටාක, ජල මාර්ග, මණ්ඩප, අංගන සහ සැලසුම් කළ උද්‍යාන දැකගත හැකිය.

SINLLAMA V1: පුරාණ ශ්‍රී ලංකාවේ වාරිමාර්ග තාක්ෂණය ඉතා දියුණු මට්ටමක පැවතුනි.පුරාණ වැව් තැනීමේ දී ඒවා පිරවීම පිණිස යොදාගත් ක්‍රමෝපායන් ද ඒ හා සමාන ලෙස වැදගත් වේ.වැවක් ඉදිකිරීමේදී එහි හැඩය තීරණය කිරීම කෙරෙහි වැඩි අවධානයක් යොමු කරන ලදී.ඒවා අතරින් වඩාත් ප්‍රචලිත වූයේ "වටා වැව" යන නමයි.මේඅතර වට දාර සහිත විශාල ජලාශවලට වඩා කුඩා ප්‍රමාණයේ රවුම් හැඩැති ජලාශයන් පිළිබඳවද සඳහන් වී තිබේ.එහි ප්‍රතිඵලයක් වශයෙන් බොහෝ විට දක්නට ලැබුණේ තරමක් දුරට පැතලි මතුපිටකින් යුත් ජලාශ ය.කෙසේ වෙතත්

SINLLAMA V2: පුරාණ ශ්‍රී ලංකාවේ වාරිමාර්ග තාක්ෂණය පිළිබඳ හොඳම උදාහරණය ලෙස සැලකෙන්නේ අනුරාධපුර රාජධානියයි (ක්‍රිස්තු පූර්ව දෙවන සියවස සිට ක්‍රිස්තු වර්ෂ අටවන සියවසේ අවසානය දක්වා).අනුරාධපුර රාජධානියේ දී නිර්මාණය කරන ලද විශාලතම ජලාශය පරාක්‍රම සමුද්‍රයයි ("අහසින් වැටෙන එකදු වතුර බිඳක් හෝ මින

Experiment A — Test SinLlama without graph retrieval

In [8]:

import torch

test_context = (
    "සීගිරියේ ජල උද්‍යානවල ජල තටාක, "
    "ජල මාර්ග, මණ්ඩප සහ අංගන ඇත."
)

test_question = "සීගිරියේ ජල උද්‍යානවල දැකගත හැකි අංග මොනවාද?"

prompt = f"""පාඨය:
{test_context}

ප්‍රශ්නය: {test_question}
පිළිතුර:"""

inputs = tok(prompt, return_tensors="pt")
inputs = {
    key: value.to(next(llm.parameters()).device)
    for key, value in inputs.items()
}

with torch.inference_mode():
    outputs = llm.generate(
        **inputs,
        max_new_tokens=70,
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tok.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tok.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("CONTEXT:", test_context)
print("\nQUESTION:", test_question)
print("\nSINLLAMA ANSWER:", answer)


CONTEXT: සීගිරියේ ජල උද්‍යානවල ජල තටාක, ජල මාර්ග, මණ්ඩප සහ අංගන ඇත.

QUESTION: සීගිරියේ ජල උද්‍යානවල දැකගත හැකි අංග මොනවාද?

SINLLAMA ANSWER: ජලය බැස යන දිය අගල් (ජල මාර්ග), ඒවා දෙපස ඇති පොකුණු හෝ කුඩා වැව් මෙන්ම ඒවාට පිවිසෙන පියගැට පෙළවල් ද ඒ අතර වේ.මෙම ලිපිය 2020 අප්‍රේල් මස9වන දින ඉංග්‍රීසි භාෂාවෙන් පලවින.Tags: කොවිඩ්19, ප්‍රජාතන්ත්‍රවාදය, මානව හිමිකම්, ශ්‍රී ලංකාවPrevious post කොරෝනා වෛරසය


Experiment B — Check what evidence SinLlama actually receives

In [9]:

print("QUESTION:")
print(payload.get("question"))

print("\nSTATUS:")
print(payload.get("status"))

print("\nEXTRACTIVE ANSWER:")
print(payload.get("extractive_answer"))

print("\nRETRIEVED EVIDENCE:")
for i, chunk in enumerate(payload.get("retrieved_chunks", []), 1):
    print(f"\n--- CHUNK {i} ---")
    print("ID:", chunk.get("id"))
    print("Heading:", chunk.get("heading"))
    print("Text:", chunk.get("text", "")[:1000])


QUESTION:
සීගිරියේ ජල උද්‍යානවල ජල මාර්ග ගැන විස්තර කරන්න.

STATUS:
SOURCE_GROUNDED_EXTRACTIVE

EXTRACTIVE ANSWER:
සීගිරි පුරාවිද්‍යා සංකීර්ණයට ඇතුළු වූ පසු දැකගත හැකි ප්‍රධාන අංගයක් වන්නේ ජල උද්‍යානයි. මෙහි ජල තටාක, ජල මාර්ග, මණ්ඩප, අංගන සහ සැලසුම් කළ උද්‍යාන දැකගත හැකිය.

RETRIEVED EVIDENCE:

--- CHUNK 1 ---
ID: sigiriya_006
Heading: 6. ජල උද්‍යාන
Text: සීගිරි පුරාවිද්‍යා සංකීර්ණයට ඇතුළු වූ පසු දැකගත හැකි ප්‍රධාන අංගයක් වන්නේ ජල උද්‍යානයි.

මෙහි ජල තටාක, ජල මාර්ග, මණ්ඩප, අංගන සහ සැලසුම් කළ උද්‍යාන දැකගත හැකිය.

ජලය පාලනය සහ බෙදාහැරීම සඳහා පැරණි කාලයේ භූගත ජල මාර්ග සහ වෙනත් ජල පද්ධති භාවිත කර ඇත.

මෙම උද්‍යාන කාශ්‍යප රජුගේ යුගයේ පැවති උසස් භූ දර්ශන නිර්මාණ සහ ඉංජිනේරු දැනුම පෙන්වයි.

--- CHUNK 2 ---
ID: sigiriya_007
Heading: 7. දිය උල්පත් උද්‍යානය
Text: සීගිරියේ ප්‍රධාන මාර්ගය දෙපස සමමිතිකව ඉදිකර ඇති දිය උල්පත් දැකගත හැකිය.

මෙම දිය උල්පත් වෙත භූගත ජල මාර්ග ඔස්සේ ජලය ලබාදී ඇත.

ගුරුත්වාකර්ෂණය සහ ජල පීඩනය භාවිත කරමින් මෙම පද්ධතිය ක්‍රියාත්මක වූ බව සැලකේ.

ප්‍රමාණවත් ජලය පවතින අවස්ථාවලද

Experiment C — Evidence-only answering with SinLlama

In [10]:

import torch

# Select only the two relevant chunks
relevant_ids = {"sigiriya_006", "sigiriya_007"}

relevant_chunks = [
    chunk
    for chunk in payload["retrieved_chunks"]
    if chunk["id"] in relevant_ids
]

context = "\n\n".join(
    chunk["text"].strip()
    for chunk in relevant_chunks
)

question = payload["question"]

print("SELECTED CHUNKS:", [c["id"] for c in relevant_chunks])
print("\nQUESTION:", question)
print("\nCONTEXT:\n", context)


SELECTED CHUNKS: ['sigiriya_006', 'sigiriya_007']

QUESTION: සීගිරියේ ජල උද්‍යානවල ජල මාර්ග ගැන විස්තර කරන්න.

CONTEXT:
 සීගිරි පුරාවිද්‍යා සංකීර්ණයට ඇතුළු වූ පසු දැකගත හැකි ප්‍රධාන අංගයක් වන්නේ ජල උද්‍යානයි.

මෙහි ජල තටාක, ජල මාර්ග, මණ්ඩප, අංගන සහ සැලසුම් කළ උද්‍යාන දැකගත හැකිය.

ජලය පාලනය සහ බෙදාහැරීම සඳහා පැරණි කාලයේ භූගත ජල මාර්ග සහ වෙනත් ජල පද්ධති භාවිත කර ඇත.

මෙම උද්‍යාන කාශ්‍යප රජුගේ යුගයේ පැවති උසස් භූ දර්ශන නිර්මාණ සහ ඉංජිනේරු දැනුම පෙන්වයි.

සීගිරියේ ප්‍රධාන මාර්ගය දෙපස සමමිතිකව ඉදිකර ඇති දිය උල්පත් දැකගත හැකිය.

මෙම දිය උල්පත් වෙත භූගත ජල මාර්ග ඔස්සේ ජලය ලබාදී ඇත.

ගුරුත්වාකර්ෂණය සහ ජල පීඩනය භාවිත කරමින් මෙම පද්ධතිය ක්‍රියාත්මක වූ බව සැලකේ.

ප්‍රමාණවත් ජලය පවතින අවස්ථාවලදී පැරණි ජල පද්ධතියේ ඇතැම් කොටස් අදත් ක්‍රියාත්මක විය හැකිය.


In [11]:

prompt = f"""ප්‍රශ්න සහ පිළිතුරු:

ප්‍රශ්නය: සීගිරියේ ජල උද්‍යානවල ඇති අංග මොනවාද?
පිළිතුර: සීගිරියේ ජල උද්‍යානවල ජල තටාක, ජල මාර්ග, මණ්ඩප සහ අංගන දැකගත හැකිය.

ප්‍රශ්නය: සීගිරියේ දිය උල්පත්වලට ජලය ලබා දෙන්නේ කෙසේද?
පිළිතුර: භූගත ජල මාර්ග ඔස්සේ දිය උල්පත්වලට ජලය ලබා දේ.

මූලාශ්‍ර තොරතුරු:
{context}

ප්‍රශ්නය: {question}
පිළිතුර:"""

inputs = tok(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
)

inputs = {
    k: v.to(next(llm.parameters()).device)
    for k, v in inputs.items()
}

with torch.inference_mode():
    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tok.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("\nQUESTION:", question)
print("\nSINLLAMA V3 ANSWER:", answer)



QUESTION: සීගිරියේ ජල උද්‍යානවල ජල මාර්ග ගැන විස්තර කරන්න.

SINLLAMA V3 ANSWER: පුරාණ ශ්‍රී ලංකාවේ වාරිමාර්ග තාක්ෂණය පිළිබඳ හොඳම උදාහරණය ලෙස සැලකෙන්නේ අනුරාධපුරය නගරයයි.එහි පිහිටි ඇලවල් 16 ක් මගින් එකල කෘෂිකර්මාන්තය දියුණු කිරීමට කටයුතු යොදා තිබුණි.එමෙන්ම එම ඇළ පද්ධතීන් හරහාම නගර නිර්මාණය කිරීම ද සිදු වී තිබේ..ඒ අනුව බලන කල මෙරට තුළ අතීතයේ දී ඉතා ඉහළ තාක්ෂණික මට්ටමකින් යුතු කෘෂිකාර්මික පද්ධතියක් පැවතුණු බවට සාක්ෂි පවතී…මේ අතර තවත් වැදගත් කරුණක් වනුයේ වර්තමානයේදී පවා අප රටෙහි දක්නට ලැබෙන ස්වභාවික ජලාශ උපයෝගී කරගනිමින් කෘෂි කර්මාන්තය නගා සිටුවීමට


Version	Approach	Result
V1-Original prompt	    -Failed


V2-Stronger Sinhala instructions	          -Failed


V3-Few-shot prompting + relevant chunks	        -Failed



In [12]:

print("Model:", llm.config.name_or_path)
print("Tokenizer vocabulary:", len(tok))
print("Model vocabulary:", llm.get_input_embeddings().weight.shape[0])
print("Chat template:", tok.chat_template)
print("EOS token:", tok.eos_token)
print("PAD token:", tok.pad_token)

Model: SAWithanage/SinLlama-Llama-3-8B-Merged
Tokenizer vocabulary: 139336
Model vocabulary: 139336
Chat template: None
EOS token: <|end_of_text|>
PAD token: <|end_of_text|>


In [13]:
import torch

source_answer = payload["extractive_answer"]
question = payload["question"]

prompt = f"""Context: {source_answer}

Question: {question}
Answer:"""

inputs = tok(prompt, return_tensors="pt")
inputs = {
    k: v.to(next(llm.parameters()).device)
    for k, v in inputs.items()
}

with torch.inference_mode():
    outputs = llm.generate(
        **inputs,
        max_new_tokens=75,
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id
    )

answer = tok.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("QUESTION:", question)
print("\nSOURCE ANSWER:", source_answer)
print("\nSINLLAMA V4:", answer)

QUESTION: සීගිරියේ ජල උද්‍යානවල ජල මාර්ග ගැන විස්තර කරන්න.

SOURCE ANSWER: සීගිරි පුරාවිද්‍යා සංකීර්ණයට ඇතුළු වූ පසු දැකගත හැකි ප්‍රධාන අංගයක් වන්නේ ජල උද්‍යානයි. මෙහි ජල තටාක, ජල මාර්ග, මණ්ඩප, අංගන සහ සැලසුම් කළ උද්‍යාන දැකගත හැකිය.

SINLLAMA V4: පුරාණ ශ්‍රී ලංකාවේ පැවති වාරිමාර්ග ක්‍රමවලින් එකක් ලෙස මෙම ජල මාර්ග හැඳින්විය හැක.මෙම ජල මාර්ග නිර්මාණය කර ඇත්තේ ජලය ගලා ඒම සඳහා වන අතර ඒවා සාමාන්‍යයෙන් ගල් හෝ මැටි වලින් සාදා ඇත.සීගිරිය වැනි ස්ථාන වල දක්නට ලැබෙන දිය අගල් ද මේ හා සමාන වේ.එනම් ඒවායේ අරමුණ වූයේ ජලය බැස යාමයි.නමුත් වර්තමානයේදී මේවා බොහෝ විට භාවිතා කරනුයේ අලංකාරය උදෙසා
